In [ ]:
# ============================================
# 1. IMPORT REQUIRED LIBRARIES
# ============================================

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

In [ ]:
# ============================================
# 2. VISUALIZATION CONFIGURATION
# ============================================

sns.set_theme(style="whitegrid")

plt.rcParams["figure.figsize"] = (8, 6)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [ ]:
# ============================================
# 3. LOAD DATA
# ============================================

DATA_PATH = Path("Telco-Customer-Churn.csv")

df = pd.read_csv(DATA_PATH)

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
df.shape

In [ ]:

df.columns.to_list()

In [ ]:
df.sample(5)

In [ ]:
df.info()

In [ ]:
df.describe().round(2)

In [ ]:
# ============================================
# 4. BASIC DATA AUDIT
# ============================================

print("Shape:", df.shape)

print("\nDuplicate rows:", df.duplicated().sum())

print("\nMissing values:")
print(df.isnull().sum())

print("\nData types:")
print(df.dtypes)

In [ ]:
# ============================================
# MISSING VALUE REPORT
# ============================================

missing_report = pd.DataFrame({
    "missing_count": df.isnull().sum(),
    "missing_percentage": df.isnull().mean() * 100
})

missing_report = (
    missing_report
    .sort_values("missing_percentage", ascending=False)
)

missing_report

In [ ]:
#Duplicate Analysis
duplicate_count = df.duplicated().sum()

print(f"Number of duplicate rows: {duplicate_count}")

In [ ]:
#Unique Values / Cardinality:

df.nunique().sort_values()

In [ ]:
df['TotalCharges'] = pd.to_numeric(
    df['TotalCharges'].str.strip(),
    errors='coerce'
).fillna(0)

## Target Variable Analysis

In [ ]:
TARGET = 'Churn'

df[TARGET].value_counts()

In [1]:
df[TARGET].valuecounts(normalize=True) * 100

NameError: name 'df' is not defined

In [ ]:
sns.countplot(data=df, x=TARGET)

plt.title("Customer Churn Distribution")
plt.xlabel("Churn")
plt.ylabel("Number of Customers")

plt.show()

In [ ]:
numerical_features = df.select_dtypes(include=['int64','float64']).columns

categorical_features = df.select_dtypes(include=["object", "category", "bool"]).columns

In [ ]:
print("Numerical Features:")
print(numerical_features.tolist())

print("\nCategorical Features:")
print(categorical_features.tolist())

## Numerical Distribution

In [ ]:
df["tenure"].describe()

sns.histplot(
    data=df,
    x="tenure",
    kde=True
)

plt.title("Tenure Distribution")
plt.show()

## Boxpplot

In [ ]:
sns.boxplot(
    data=df,
    x="tenure"
)

plt.title("Tenure Boxplot")
plt.show()

## Analyze All Numerical Variables

In [ ]:
for feature in numerical_features:

    plt.figure(figsize=(8, 5))

    sns.histplot(
        data=df,
        x=feature,
        kde=True
    )

    plt.title(f"Distribution of {feature}")
    plt.show()

## Categorical Analysis

In [ ]:
for feature in categorical_features:

    if feature == TARGET:
        continue

    churn_rate = (
        df.groupby(feature)[TARGET]
        .apply(lambda x: (x == "Yes").mean() * 100)
        .sort_values(ascending=False)
    )

    print(f"\n{'=' * 60}")
    print(f"Feature: {feature}")
    print(churn_rate)

## Correlation Analysis

In [ ]:
correlation_matrix = df[numerical_features].corr()

correlation_matrix

In [ ]:
plt.figure(figsize=(10, 7))

sns.heatmap(
    correlation_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm"
)

plt.title("Numerical Feature Correlation")

plt.show()

## Outlier Analysis

In [ ]:
def detect_outliers_iqr(data, feature):
    """
    Detect potential outliers using the IQR method.
    """

    Q1 = data[feature].quantile(0.25)
    Q3 = data[feature].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = data[
        (data[feature] < lower_bound) | (data[feature] > upper_bound)
    ]

    return outliers

In [ ]:
outliers = detect_outliers_iqr(df, "MonthlyCharges")

print("Outliers:", len(outliers))

In [ ]:
df['Churn'] = df['Churn'].map({
    'Yes':1,
    'No':0
})

In [ ]:
X = df.drop(columns=['customerID', 'Churn'])
y = df['Churn']


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42, stratify=y)

In [ ]:
numerical_features = X_train.select_dtypes(include=['int64','float64']).columns

categorical_features = X_train.select_dtypes(include=['object']).columns


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

In [ ]:
num_pipeline = Pipeline([
    ('Scalar', StandardScaler())
])

cat_pipeline = Pipeline([
    ('Encoder', OneHotEncoder(handle_unknown='ignore',sparse_output=False))
])

In [ ]:
preprocessor = ColumnTransformer([
    ('num', num_pipeline, numerical_features ),
    ('cat', cat_pipeline, categorical_features)
])

In [ ]:
X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

In [ ]:
print("X_train_shape:",X_train_transformed.shape)
print("X_train_shape:",X_test_transformed.shape)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=20,random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=20, random_state=42),
    "SVM": SVC(),
    "XGB":XGBClassifier(n_estimators=100, random_state=42)
}

In [ ]:
from sklearn.model_selection import cross_validate

cv_results = []

for name, model in models.items():

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=5,
        scoring=["accuracy", "precision", "recall", "f1"],
        n_jobs=-1
    )

    cv_results.append({
        "Model": name,
        "Accuracy": scores["test_accuracy"].mean(),
        "Precision": scores["test_precision"].mean(),
        "Recall": scores["test_recall"].mean(),
        "F1": scores["test_f1"].mean()
    })

In [ ]:
baseline_df = pd.DataFrame(cv_results)

print(baseline_df.sort_values(
    by="F1",
    ascending=False 
))

In [ ]:
promising_models = {
    "Logistic Regression": models["Logistic Regression"],
    "Gradient Boosting": models["Gradient Boosting"],
}

In [ ]:
models_to_tune = {
    "Logistic Regression": {
                            "model": LogisticRegression(max_iter=1000),
                            "params": {
                                        "model__C": [0.01, 0.1, 1, 10, 100],
                                        "model__solver": ["liblinear", "lbfgs"],
                                        "model__class_weight": [None, "balanced"]
        }
    },

    "Gradient Boosting": {
                        "model": GradientBoostingClassifier(random_state=42),
                        "params": {
                                    "model__n_estimators": [50, 100, 150, 200],
                                    "model__learning_rate": [0.01, 0.05, 0.1, 0.2],
                                    "model__max_depth": [2, 3, 4, 5],
                                    "model__min_samples_split": [2, 5, 10],
                                    "model__min_samples_leaf": [1, 2, 4],
                                    "model__subsample": [0.8, 1.0]
                }
    }
}

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

tuned_models = {}
tuning_results = []

for name, config in models_to_tune.items():

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", config["model"])
    ])

    search = RandomizedSearchCV(
        estimator=pipeline,
        param_distributions=config["params"],
        n_iter=20,
        scoring="f1",
        cv=5,
        random_state=42,
        n_jobs=-1
    )

    search.fit(X_train, y_train)

    tuned_models[name] = search

    tuning_results.append({
        "Model": name,
        "Best CV F1": search.best_score_,
        "Best Parameters": search.best_params_
    })

In [109]:
tuning_df = pd.DataFrame(tuning_results)

print(tuning_df.sort_values(
    "Best CV F1",
    ascending=False
))

                 Model  Best CV F1  \
0  Logistic Regression    0.626873   
1    Gradient Boosting    0.588340   

                                     Best Parameters  
0  {'model__solver': 'liblinear', 'model__class_w...  
1  {'model__subsample': 1.0, 'model__n_estimators...  


In [110]:
best_model_name = tuning_df.iloc[0]["Model"]

best_search = tuned_models[best_model_name]

print("Best Model:", best_model_name)
print("Best CV F1:", best_search.best_score_)
print("Best Parameters:", best_search.best_params_)

Best Model: Logistic Regression
Best CV F1: 0.6268725146789222
Best Parameters: {'model__solver': 'liblinear', 'model__class_weight': 'balanced', 'model__C': 100}


In [115]:
from sklearn.metrics import (f1_score,confusion_matrix,classification_report)

In [112]:
final_results = []

for name, search in tuned_models.items():

    y_pred = search.best_estimator_.predict(X_test)

    final_results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred)
    })

final_df = pd.DataFrame(final_results)

print(final_df.sort_values(
    by="F1",
    ascending=False
))

                 Model  Accuracy  Precision    Recall        F1
0  Logistic Regression  0.742796   0.509804  0.800648  0.622951
1    Gradient Boosting  0.802581   0.676339  0.491086  0.569014


## Logistic Regression:

Recall = 0.8006

That means your model is catching about 80% of the actual churners.